In [9]:
import torch 
import os
import numpy as np
import torch.nn.functional as F

from torch.utils.data import Dataset, random_split
from torch import nn
from torchvision import datasets, transforms

In [10]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


In [11]:
# download dataset and split train and test set

cf10_training_data = datasets.CIFAR10(root = 'data', train = True, download= True, transform= transforms.ToTensor()) 

cf10_test_data = datasets.CIFAR10(root = 'Data', train = False, download= True, transform = transforms.ToTensor())


cf100_training_data = datasets.CIFAR100(root = 'data', train = True, download= True, transform= transforms.ToTensor()) 

cf100_test_data = datasets.CIFAR100(root = 'Data', train = False, download= True, transform = transforms.ToTensor())

In [12]:
# create validation set
print(len(cf10_training_data))


# print(len(cf100_training_data))

cf10_val_data = random_split(cf10_training_data, [int(len(cf10_training_data)*0.8), int(len(cf10_training_data)*0.2)])
#cf100_val_data = random_split(cf100_training_data, [len(cf100_training_data)*0.8, len(cf100_training_data)*0.2])

50000


In [13]:
print(1)
training_loader = torch.utils.data.DataLoader(cf10_training_data, batch_size=32, shuffle=True)
validation_loader = torch.utils.data.DataLoader(cf10_val_data, batch_size=32, shuffle=True)
testing_loader = torch.utils.data.DataLoader(cf10_test_data, batch_size=32, shuffle=False)

1


In [14]:
class LesNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3,6,5,1)
        self.conv2 = nn.Conv2d(6, 16, 5, 1)

        self.fc1 = nn.Linear(5*5*16, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

        for m in self.modules():
            if isinstance(m, nn.Conv2d) or isinstance(m, nn.Linear):
                nn.init.kaiming_uniform_(m.weight, mode='fan_in', nonlinearity='relu')

    def forward(self, X):
        X = F.relu(self.conv1(X))
        X = F.max_pool2d(X, (2,2))
        X = F.relu(self.conv2(X))
        X = F.max_pool2d(X, (2,2))
        X = X.flatten(1)

        X = F.relu(self.fc1(X))
        X = F.relu(self.fc2(X))
        X = self.fc3(X)
        return F.log_softmax(X, dim=1)
        

In [15]:
torch.manual_seed(42)
print(1)
model_1 = LesNet().to(device)

1


In [16]:
def training_model(model, nr_epochs):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    losses = []
    for epoch in range(nr_epochs):
        running_loss = 0

        for i , data in enumerate(training_loader):
            inputs, label = data

            if torch.cuda.is_available():
                inputs, label = inputs.cuda(), label.cuda()
                model.cuda()

            else:
                model.cpu()

            optimizer.zero_grad()
            outputs = model.forward(inputs)
            loss = criterion(outputs, label)

            

            loss.backward()
            optimizer.step()

            running_loss += loss.item()

           
        print(f"Epoch {epoch+1}, Loss : {running_loss/len(training_loader)}")


            
training_model(model_1, 20)

Epoch 1, Loss : 1.5909495580981956
Epoch 2, Loss : 1.311343312377893
Epoch 3, Loss : 1.1889776874259734
Epoch 4, Loss : 1.1112667764903488
Epoch 5, Loss : 1.0447438185175335
Epoch 6, Loss : 0.9925119229333186
Epoch 7, Loss : 0.9420878545298305
Epoch 8, Loss : 0.8964326164315163
Epoch 9, Loss : 0.8586707376015164
Epoch 10, Loss : 0.8229297344606806
Epoch 11, Loss : 0.7851103284918797
Epoch 12, Loss : 0.7515348414255881
Epoch 13, Loss : 0.7228992198494406
Epoch 14, Loss : 0.6927894653796539
Epoch 15, Loss : 0.6632322844544513
Epoch 16, Loss : 0.6374187318659409
Epoch 17, Loss : 0.6154716556566462
Epoch 18, Loss : 0.5870599982662988
Epoch 19, Loss : 0.5754415538741164
Epoch 20, Loss : 0.5489631564054288


In [17]:
# model 2

class Model2(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3,6,5,1)
        self.conv2 = nn.Conv2d(6, 16, 5, 1)

        self.fc1 = nn.Linear(5*5*16, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

        #add dropuot 
        #self.dropout = nn.Dropout2d(p = 0.3) # 30% chance to be 0

        for m in self.modules():
            if isinstance(m, nn.Conv2d) or isinstance(m, nn.Linear):
                nn.init.kaiming_uniform_(m.weight, mode='fan_in', nonlinearity='relu') #different relu

    def forward(self, X):
        X = F.relu(self.conv1(X))
        X = F.max_pool2d(X, (2,2))
        X = F.dropout2d(X, p = 0.3)  #add dropuot 

        X = F.relu(self.conv2(X))
        X = F.max_pool2d(X, (2,2))
        X = F.dropout2d(X, p = 0.3)  #add dropuot 
        
        X = X.flatten(1)

        X = F.relu(self.fc1(X))
        X = F.relu(self.fc2(X))
        X = self.fc3(X)
        return F.log_softmax(X, dim=1)

model_2 = Model2().to(device)

In [18]:
# model 3

class Model3(nn.Module): #Batch Normalization
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3,6,5,1)
        self.bn1 = nn.BatchNorm2d(6)
        self.conv2 = nn.Conv2d(6, 16, 5, 1)
        self.bn2 = nn.BatchNorm2d(16)
        

        self.fc1 = nn.Linear(5*5*16, 120)
        self.bn3 = nn.BatchNorm1d(120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

      

        for m in self.modules():
            if isinstance(m, nn.Conv2d) or isinstance(m, nn.Linear):
                nn.init.kaiming_uniform_(m.weight, mode='fan_in', nonlinearity='relu') #different relu

    def forward(self, X):
        X = F.relu(self.bn1(self.conv1(X)))
        X = F.max_pool2d(X, (2,2))
        

        X = F.relu(self.bn2(self.conv2(X)))
        X = F.max_pool2d(X, (2,2))
       

        X = X.flatten(1)

        X = F.relu(self.bn3(self.fc1(X)))
        X = F.relu(self.fc2(X))
        X = self.fc3(X)
        return F.log_softmax(X, dim=1)

model_3 = Model3().to(device)

In [20]:
training_model(model_2, 20)

Epoch 1, Loss : 1.9457300950988163
Epoch 2, Loss : 1.7154646925032329
Epoch 3, Loss : 1.6407980722871562
Epoch 4, Loss : 1.5899477113307627
Epoch 5, Loss : 1.5561189440222634
Epoch 6, Loss : 1.5340828192745046
Epoch 7, Loss : 1.5114206989377412
Epoch 8, Loss : 1.4890255453643018
Epoch 9, Loss : 1.4694530225989915
Epoch 10, Loss : 1.456793706957072
Epoch 11, Loss : 1.4465530428151174
Epoch 12, Loss : 1.423924316066393
Epoch 13, Loss : 1.4156321527022853
Epoch 14, Loss : 1.4033721133980779
Epoch 15, Loss : 1.3881600923211774
Epoch 16, Loss : 1.3837367274863401
Epoch 17, Loss : 1.375244446427717
Epoch 18, Loss : 1.3626245465785056
Epoch 19, Loss : 1.3527111247115156
Epoch 20, Loss : 1.3479365501278726


In [21]:
# testing model 2



training_model(model_3, 20)

Epoch 1, Loss : 1.497281255168329
Epoch 2, Loss : 1.2160629011542845
Epoch 3, Loss : 1.0967504964070067
Epoch 4, Loss : 1.014731656414381
Epoch 5, Loss : 0.9510049170701838
Epoch 6, Loss : 0.901539318411303
Epoch 7, Loss : 0.85918597046641
Epoch 8, Loss : 0.8227917250157623
Epoch 9, Loss : 0.7892679136987687
Epoch 10, Loss : 0.7631699200040319
Epoch 11, Loss : 0.7405429953233752
Epoch 12, Loss : 0.7156283360830271
Epoch 13, Loss : 0.6878181112571473
Epoch 14, Loss : 0.6726984555760943
Epoch 15, Loss : 0.6562213740589827
Epoch 16, Loss : 0.6373748301372876
Epoch 17, Loss : 0.6229667050081114
Epoch 18, Loss : 0.606535907269897
Epoch 19, Loss : 0.5939455581107966
Epoch 20, Loss : 0.5836378689748083
